## Problem: Implement Mixed Precision Training Using `torch.amp`

### Problem Statement
Mixed precision training runs most of the forward and backward pass in a 16-bit floating-point type (`float16` or `bfloat16`) while keeping the weights and numerically sensitive operations in `float32`. On GPUs with tensor cores this is faster and roughly halves activation memory, usually without hurting accuracy. Your task is to implement it with PyTorch's automatic mixed precision (AMP): `torch.autocast` and `torch.amp.GradScaler`.

> `torch.cuda.amp.autocast()` and `torch.cuda.amp.GradScaler()` are deprecated aliases. Use `torch.autocast("cuda", ...)` and `torch.amp.GradScaler("cuda")` instead.

### Background
- **What autocast does.** It picks a dtype per operation. Matmuls, convolutions and `nn.Linear` run in the low-precision dtype. Reductions, softmax, normalization layers and losses such as `mse_loss` run in `float32`. The parameters themselves stay `float32` (the "master weights"), so small optimizer updates are not lost to rounding. Wrap only the forward pass and the loss: backward ops automatically run in the same dtype as their forward ops.
- **float16 vs bfloat16.**
  - `float16` has 5 exponent bits and 10 mantissa bits. Its largest value is 65504, its smallest normal value is about 6.1e-5, and its subnormals go down to about 6e-8. Small gradients underflow to zero, so float16 needs **loss scaling**.
  - `bfloat16` has 8 exponent bits, giving it the same range as `float32`, and 7 mantissa bits. Its precision is coarser, but underflow is not a problem, so it needs **no** `GradScaler`. It requires hardware support (NVIDIA Ampere or newer, recent CPUs).
- **GradScaler** (for float16 only):
  - `scaler.scale(loss)` multiplies the loss by a factor S (initially 2¹⁶), so the gradients computed by `backward()` stay representable in float16.
  - `scaler.step(optimizer)` divides the gradients by S. If any gradient is inf or NaN, it skips the update.
  - `scaler.update()` halves S after an overflow and doubles it after 2000 consecutive clean steps.
  - If you clip gradients, call `scaler.unscale_(optimizer)` first, so the clipping threshold applies to the true gradients rather than the scaled ones.

### Requirements
1. Use `float16` on CUDA and `bfloat16` on CPU, so the notebook runs anywhere. Create a `torch.amp.GradScaler` that is enabled only for `float16`.
2. Run the forward pass and the loss under `torch.autocast(device_type=..., dtype=...)`.
3. Run backward through the scaler:
   - `scaler.scale(loss).backward()`
   - `scaler.unscale_(optimizer)`, then gradient clipping
   - `scaler.step(optimizer)`
   - `scaler.update()`
4. Run inference under `torch.autocast` with no scaler, since there are no gradients to scale.

In [ ]:
# Implement mixed precision training in PyTorch using torch.amp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)


device = "cuda" if torch.cuda.is_available() else "cpu"

# Generate synthetic data with a learnable (linear + noise) target
torch.manual_seed(0)
X = torch.randn(1000, 10)
y = X @ torch.randn(10, 1) + 0.1 * torch.randn(1000, 1)
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model, loss function, and optimizer
model = SimpleModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# float16 on GPU needs loss scaling; bfloat16 has float32's exponent range and does not
amp_dtype = torch.float16 if device == "cuda" else torch.bfloat16
scaler = torch.amp.GradScaler(device, enabled=(amp_dtype == torch.float16))

In [ ]:
# Checks: what autocast changes, and what it leaves alone
x = torch.randn(4, 10, device=device)
with torch.autocast(device_type=device, dtype=amp_dtype):
    out = model(x)
    check_loss = criterion(out, torch.zeros(4, 1, device=device))
assert out.dtype == amp_dtype, f"nn.Linear should run in {amp_dtype} under autocast, got {out.dtype}"
assert check_loss.dtype == torch.float32, "mse_loss is autocast to float32"
assert model.fc.weight.dtype == torch.float32, "the parameters (master weights) stay float32"
assert scaler.is_enabled() == (amp_dtype == torch.float16), "loss scaling is only needed for float16"
print("All checks passed")

In [ ]:
# Training loop
epochs = 5
for epoch in range(epochs):
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass under autocast: the Linear runs in amp_dtype, mse_loss in float32
        with torch.autocast(device_type=device, dtype=amp_dtype):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # Backward pass outside autocast, with scaled gradients
        optimizer.zero_grad()
        scaler.scale(loss).backward()  # grads of loss * S, so small float16 grads don't underflow
        scaler.unscale_(optimizer)     # divide by S first, so clipping sees the true gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)         # skipped if any gradient is inf/NaN
        scaler.update()                # shrink S after an overflow, grow it after a clean streak

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

In [ ]:
# Inference: autocast without a scaler, since there are no gradients
model.eval()
X_test = torch.randn(5, 10, device=device)
with torch.inference_mode(), torch.autocast(device_type=device, dtype=amp_dtype):
    predictions = model(X_test)
print("Predictions:", predictions.float().squeeze(1), f"(computed in {predictions.dtype})")